In [0]:
import json, os
from pyspark.sql import functions as F, Window
CATALOG= "hawkerreach"
dbutils.widgets.text("target_schema", "gold")  # dev_<name> for test runs
OUT= f"{CATALOG}.{dbutils.widgets.get('target_schema')}"
print("writing to", OUT)

In [0]:
# all model values come from assumptions.json at the repo root (shared with the app); pull the Git folder before each run
if not os.path.exists("assumptions.json"):
    raise FileNotFoundError(f"assumptions.json not found in {os.getcwd()}: run this notebook from a Git folder clone of the repo")
with open("assumptions.json") as f:
    A= {k: v["value"] for k, v in json.load(f).items()}

EATING_SHARE= A["diners.eating_share"]              # 0.83, NEA Perception Survey 2018
VISITS_BASE= A["diners.visits_per_week.base"]       # base case 3
UNDERCOUNT_FACTOR= A["undercount_factor"]           # share of median residents per stall
SENIOR_PACE= A["walk.pace.senior"]                  # 1.0 m/s
LIMIT_MIN= A["walk.limit_min"]                      # 15 min
DINERS_PER_RESIDENT= EATING_SHARE * VISITS_BASE / 7
print(f"eating share {EATING_SHARE} | visits/week {VISITS_BASE} | diners per resident per day {DINERS_PER_RESIDENT:.4f} | undercount factor {UNDERCOUNT_FACTOR} | senior pace {SENIOR_PACE} m/s | limit {LIMIT_MIN} min")

In [0]:
# every centre within 2 km of each block, ranked; ties broken by centre_id (04_gold_access sorts by walk_m only, so block_access v2 needs this tie-break too)
w= Window.partitionBy("block_id").orderBy("walk_m", "centre_id")
ranked= spark.table(f"{CATALOG}.silver.dist_matrix").withColumn("rank", F.row_number().over(w))

subzones= spark.table(f"{CATALOG}.silver.subzones").select("subzone_n", F.col("subzone_c").alias("subzone_id"))
blocks= (spark.table(f"{CATALOG}.silver.blocks").join(subzones, "subzone_n", "left")  # blocks carry only the subzone name
         .select("block_id", "lon", "lat", "subzone_id", "est_residents", "est_seniors"))
options= (blocks.join(ranked, "block_id", "left")  # a block with no centre within 2 km keeps one row with centre_id null
    .withColumn("diners_per_day", F.col("est_residents") * DINERS_PER_RESIDENT)
    .select("block_id", "centre_id", "walk_m", "rank", "lon", "lat", "subzone_id", "est_residents", "est_seniors", "diners_per_day"))

options.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.block_options")

In [0]:
# step row counts go to data_quality_log once Spec 1 builds it (ASSUMPTIONS.md, Still open); until then they are printed
o= spark.table(f"{OUT}.block_options")
n_blocks= spark.table(f"{CATALOG}.silver.blocks").count()
no_centre= o.filter(F.col("centre_id").isNull())
print("rows:", o.count(), "| blocks:", o.select("block_id").distinct().count(), "of", n_blocks, "| blocks with no centre within 2 km:", no_centre.count())

assert o.select("block_id").distinct().count() == n_blocks, "a block is missing from block_options"
assert o.filter(F.col("subzone_id").isNull()).count() == 0, "a block has no subzone code"
assert no_centre.filter(F.col("rank").isNotNull()).count() == 0, "a no-centre row has a rank"
assert o.filter(F.col("centre_id").isNotNull() & F.col("rank").isNull()).count() == 0, "a centre row has no rank"
gaps= o.filter(F.col("centre_id").isNotNull()).groupBy("block_id").agg(F.count("*").alias("n"), F.max("rank").alias("top")).filter("n != top")
assert gaps.count() == 0, "ranks have gaps"

# ranks 1-3 must match 04_gold_access exactly (needs block_access v2 from Spec 1)
ba= spark.table(f"{CATALOG}.gold.block_access")
if "second_centre_id" in ba.columns:
    mine= (o.groupBy("block_id")
            .agg(*[F.max(F.when(F.col("rank") == r, F.col("centre_id"))).alias(c)
                   for r, c in [(1, "nearest_centre_id"), (2, "second_centre_id"), (3, "third_centre_id")]]))
    cols= ["nearest_centre_id", "second_centre_id", "third_centre_id"]
    diff= mine.alias("m").join(ba.alias("b"), "block_id").filter(" OR ".join(f"NOT (m.{c} <=> b.{c})" for c in cols))
    assert diff.count() == 0, f"{diff.count()} blocks rank their top 3 differently from block_access"
    print("ranks 1-3 match block_access")
else:
    print("skipped block_access check: block_access v2 (second_centre_id) not built yet")

In [0]:
# baseline = residents whose nearest centre it is
centres= spark.table(f"{CATALOG}.silver.centres").select("centre_id", "cooked_food_stalls")
baseline= o.filter("rank = 1").groupBy("centre_id").agg(F.sum("est_residents").alias("baseline_residents"))

demand= (centres.join(baseline, "centre_id", "left")
    .fillna(0, ["baseline_residents"])  # never the nearest centre to any block
    .withColumn("baseline_diners_per_day", F.col("baseline_residents") * DINERS_PER_RESIDENT)
    .withColumn("residents_per_stall", F.col("baseline_residents") / F.col("cooked_food_stalls")))
median= demand.agg(F.expr("percentile(residents_per_stall, 0.5)")).first()[0]
demand= (demand.withColumn("possibly_undercounted", F.col("residents_per_stall") < UNDERCOUNT_FACTOR * median)
               .select("centre_id", "baseline_residents", "baseline_diners_per_day", "residents_per_stall", "possibly_undercounted"))

demand.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.centre_demand")

In [0]:
d= spark.table(f"{OUT}.centre_demand")
baseline_total= d.agg(F.sum("baseline_residents")).first()[0]
# expected total from silver, not from block_options: residents of blocks with any centre within 2 km
reachable= spark.table(f"{CATALOG}.silver.dist_matrix").select("block_id").distinct()
served= spark.table(f"{CATALOG}.silver.blocks").join(reachable, "block_id", "left_semi").agg(F.sum("est_residents")).first()[0]
print("centres:", d.count(), "| median residents per stall:", round(median), "| baseline residents:", round(baseline_total), "of", round(served))
assert abs(baseline_total - served) < 0.01, "baseline residents don't add up to residents with a centre within 2 km"

flagged= (d.filter("possibly_undercounted")
           .join(spark.table(f"{CATALOG}.silver.centres").select("centre_id", "name", "cooked_food_stalls"), "centre_id")
           .orderBy("residents_per_stall"))
flagged_names= [r.name for r in flagged.select("name").collect()]
expected= ["Amoy Street", "Market Street", "Newton", "Adam Road", "East Coast Lagoon", "Pasir Panjang"]  # 0 modelled residents
missing= [e for e in expected if not any(e in n for n in flagged_names)]
assert not missing, f"expected these among the possibly undercounted centres: {missing}"
display(flagged)

In [0]:
# single-centre closures: each block whose nearest centre closes moves, all or nothing, to its 2nd-nearest centre
WALK_LIMIT_M= SENIOR_PACE * LIMIT_MIN * 60   # 900 m
print(f"walk limit {WALK_LIMIT_M:.0f} m")

first= o.filter("rank = 1").select("block_id", F.col("centre_id").alias("closed_centre_id"), F.col("walk_m").alias("walk_before"), "est_residents", "est_seniors")
second= o.filter("rank = 2").select("block_id", F.col("centre_id").alias("neighbour_id"), F.col("walk_m").alias("walk_after"))
moves= first.join(second, "block_id", "left")  # no 2nd centre within 2 km = newly unserved

centres= spark.table(f"{CATALOG}.silver.centres").select("centre_id", "cooked_food_stalls", "is_interim")
load= (moves.filter(F.col("neighbour_id").isNotNull())
    .groupBy("closed_centre_id", "neighbour_id").agg(F.sum("est_residents").alias("extra_residents"))
    .join(d.select(F.col("centre_id").alias("neighbour_id"), "baseline_residents"), "neighbour_id")
    .join(centres.select(F.col("centre_id").alias("neighbour_id"), "cooked_food_stalls"), "neighbour_id")
    .withColumn("extra_per_stall", F.when(F.col("cooked_food_stalls") > 0, F.col("extra_residents") * DINERS_PER_RESIDENT / F.col("cooked_food_stalls")))  # null for a centre with no stalls
    .withColumn("pct", F.when(F.col("baseline_residents") > 0, 100 * F.col("extra_residents") / F.col("baseline_residents"))))

# worst neighbour = most extra diners per stall; its % is for that same neighbour
w= Window.partitionBy("closed_centre_id").orderBy(F.desc_nulls_last("extra_per_stall"), "neighbour_id")
worst= (load.withColumn("k", F.row_number().over(w)).filter("k = 1")
        .select("closed_centre_id", F.col("neighbour_id").alias("worst_neighbour_id"),
                F.col("extra_per_stall").alias("worst_neighbour_extra_per_stall"), F.col("pct").alias("worst_neighbour_pct")))

lost= (F.col("walk_before") <= WALK_LIMIT_M) & (F.col("walk_after").isNull() | (F.col("walk_after") > WALK_LIMIT_M))
totals= moves.groupBy("closed_centre_id").agg(
    F.sum("est_residents").alias("residents_displaced"),
    F.sum("est_seniors").alias("seniors_displaced"),
    F.sum(F.when(F.col("neighbour_id").isNull(), F.col("est_residents")).otherwise(0)).alias("unserved_residents"),
    F.sum(F.when(lost, F.col("est_seniors")).otherwise(0)).alias("seniors_lost_walk"))

counts= ["residents_displaced", "seniors_displaced", "unserved_residents", "seniors_lost_walk"]
scenarios= (centres.filter(~F.col("is_interim"))  # interim centres take diners but can't be closed
    .select(F.col("centre_id").alias("closed_centre_id"))
    .join(totals, "closed_centre_id", "left").join(worst, "closed_centre_id", "left")
    .fillna(0, counts)  # never the nearest centre to any block: no impact
    .select(F.col("closed_centre_id").alias("centre_id"), *counts,
            "worst_neighbour_id", "worst_neighbour_extra_per_stall", "worst_neighbour_pct"))

scenarios.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.closure_scenarios")

In [0]:
s= spark.table(f"{OUT}.closure_scenarios")
names= spark.table(f"{CATALOG}.silver.centres").select("centre_id", "name")
print("closable centres:", s.count(), "| no impact:", s.filter("residents_displaced = 0").count())

# seniors who lose their walk must match Page 2 (04_gold_access) for every closure
if spark.catalog.tableExists(f"{CATALOG}.gold.access_sensitivity"):
    cut= (spark.table(f"{CATALOG}.gold.access_sensitivity").filter(F.col("limit_min") == LIMIT_MIN)
          .select(F.col("closed_centre_id").alias("centre_id"), "seniors_cut_off"))
    assert s.join(cut, "centre_id").count() == s.count(), "some closable centres are missing from access_sensitivity"
    diff= s.join(cut, "centre_id").filter(F.abs(F.col("seniors_lost_walk") - F.col("seniors_cut_off")) > 0.01)
    assert diff.count() == 0, f"{diff.count()} centres disagree with access_sensitivity.seniors_cut_off"
    print("seniors_lost_walk matches access_sensitivity")
else:
    print("SKIPPED access_sensitivity check: table not built yet; rerun after #38 merges")

def find_centre(text):
    rows= s.join(names, "centre_id").filter(F.col("name").contains(text)).collect()
    assert len(rows) == 1, f"expected one centre named like {text!r}, got {len(rows)}"
    return rows[0]

kim_keat= find_centre("Kim Keat")
kim_keat_neighbour= names.filter(F.col("centre_id") == kim_keat.worst_neighbour_id).first().name
print(f"Kim Keat Palm -> {kim_keat_neighbour}: +{kim_keat.worst_neighbour_pct:.0f}%, {kim_keat.worst_neighbour_extra_per_stall:.0f} extra diners per stall")
assert "Lorong 5 Blk 75" in kim_keat_neighbour, "Kim Keat Palm's worst neighbour should be Toa Payoh Lorong 5 Blk 75"
assert find_centre("East Coast Lagoon").residents_displaced == 0, "East Coast Lagoon should displace no one"

# shortlist for the demo anchor
display(s.join(names, "centre_id").orderBy(F.desc("worst_neighbour_extra_per_stall")).limit(10))